# Learnings from this project

## Tools
* ResNet18 - image classification

## Reading list
* [CS231n backpropagation notes](https://cs231n.github.io/optimization-2/)
* [micrograd's source](https://github.com/karpathy/micrograd)

* Nielsen, Neural Networks and Deep Learning, Chapter 2 (http://neuralnetworksanddeeplearning.com/chap2.html) — free online book. The careful derivation of the four backprop equations, with more hand-holding than the MIT chapter. Read this when you want the math properly rather than operationally.
* [CNN wikipedia page](https://en.wikipedia.org/wiki/Convolutional_neural_network)
* [read this last](https://visionbook.mit.edu/backpropagation.html)




## Workflow

1. Download data from EuroSAT
    * Data consists of satelite images
1. Split data into training, testing, validation
1. Build transformations
    * the transformations do not operate on the image itself; when the image is sent to the model, the image is transformed that pass through


### lifecycle of one image during training

1. The DataLoader asks the dataset for image #4823.
2. The dataset opens that JPEG from disk and decodes it — now there's a 64×64 array in RAM.
3. It calls the transform on that array. Resize produces a new 224×224 array. Normalize produces another array. Each step returns a new object; the previous one becomes garbage.
4. The dataset returns that final array plus the label.
5. The DataLoader does this 32 times and stacks the results into one array of shape (32, 3, 224, 224).
6. That batch goes into the model. Loss is computed. Weights are updated.
7. The batch is discarded. Nothing about it survives.
8. Next batch: read from disk again, decode again, transform again.

so the weights do keep the updated values based on the fxns and equations, but we dont need to keep the actual information once the weights are adjusted; teh weights are what tune the model

#### from claude

Setup — once, before training

1. Transforms — resize to 224, normalize with ImageNet statistics, random flips on train only.
2. Dataset — applies the transforms as each image is loaded.
3. DataLoader — one per split. Batch size 32, shuffle on train only.
4. Load pretrained weights — resnet18 with ImageNet parameters.
5. Replace fc — swap the 1000-output fully connected layer for a 10-output one, randomly initialized.
6. Freeze — requires_grad = False on every parameter except fc.
7. Loss function — cross-entropy.
8. Optimizer — Adam, given only the parameters that still require gradients.

Stage 1 — frozen backbone, ~3 epochs

Per epoch, per batch:

1. zero_grad — clear gradients left from the previous batch
2. Forward pass — batch through the network, 10 outputs per image
3. Loss — cross-entropy against the true labels
4. Backward pass — backpropagation computes gradients
5. Optimizer step — parameters updated; only fc moves, the rest are frozen

Then, once per epoch:

6. Validation pass — model.eval() and torch.no_grad(), forward only, no gradient computation and no updates
7. Record training loss, validation loss, validation accuracy
8. Checkpoint if validation accuracy improved

Stage 2 — fine-tuning, ~5 epochs

9. Unfreeze — requires_grad = True on all parameters
10. New optimizer at a lower learning rate (~1e-4), now covering every parameter
11. Repeat steps 1–8 unchanged

Evaluation — once, at the end

12. Load the best checkpoint
13. Test set forward pass under no_grad
14. Accuracy, confusion matrix, per-class precision and recall
15. Compare Stage 1 versus Stage 2 results — this quantifies what fine-tuning bought over transfer learning alone

Diagnostics throughout

Plot training and validation loss per epoch. Both falling means it's still learning. Training falling while validation rises means overfitting.

## CNN Theory

#### Terms
Parameters (weights): the numbers inside the network that get adjusted during training. ResNet18 has about 11 million. They're organized into layers — sequential stages, each transforming its input and passing it on.

Pretrained weights: a saved file of parameter values produced by training on ImageNet, a dataset of ~1.2 million labeled everyday photographs. Loading resnet18(weights=...) downloads those values instead of initializing randomly.

Fully connected layer (fc): ResNet18's final layer. It outputs 1000 values, one per ImageNet category. You replace it with a new one outputting 10. Its parameters start random because you just created it.

Forward pass: input goes through every layer in order, producing 10 output values.

Loss function: measures how wrong the outputs are versus the true label, as a single number. For multi-class classification this is cross-entropy.

Gradient: for each parameter, the derivative of the loss with respect to that parameter — how much the loss would change if you nudged it. Computed by backpropagation, which applies the chain rule backward through the layers.

Optimizer: the rule for updating parameters given their gradients. Adam is the standard choice. The learning rate scales the size of each update.

Freezing: setting requires_grad = False on a parameter. Gradients aren't computed for it and the optimizer skips it, so its value stays fixed. Freezing everything except fc means only the new layer trains.

Fine-tuning: unfreezing and training all parameters, at a lower learning rate (~1e-4) so the pretrained values shift only slightly rather than being destroyed.

Transfer learning: the general practice of starting from pretrained weights. It works because early convolutional layers learn general features — edges, textures, color gradients — that aren't specific to the original dataset.

Batch: the number of images processed per update. Batch size 32 means 32 forward passes, one loss, one backward pass, one parameter update.

Epoch: one complete pass over the training set.

Overfitting: training loss keeps falling while validation loss rises — the model is memorizing training examples rather than learning generalizable features. Detecting this is why you plot both curves.